In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxGIN)

This notebook processes and standardizes the **ToxGIN** dataset, which provides peptide sequences annotated for toxicity and originally distributed as separate training and test splits. The workflow focuses on dataset integration, duplicate resolution, and metadata generation to ensure a clean and consistent dataset suitable for downstream machine learning and benchmarking tasks.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxGIN
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads the original ToxGIN training and test datasets**:
  - both splits are treated as equally valid sources of labeled data.
- **Concatenates train and test sets** into a unified peptide toxicity dataset.
- **Standardizes sequence–label pairs** into a consistent tabular format.
- **Performs duplicate sequence checks**:
  - merges identical sequences with consistent labels,
  - flags sequences with conflicting annotations as erroneous.
- **Generates dataset-level metadata** using a centralized raw data description file.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "ToxGIN"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_test = pd.read_csv(f"{PATH_INPUT}/{name_source}/test_sequence.csv")

In [4]:
df_train = pd.read_csv(f"{PATH_INPUT}/{name_source}/train_sequence.csv")

- Concatenate dataset

In [5]:
df_toxgin = (
    pd.concat([
        df_test,
        df_train
    ], ignore_index=True)
    [["sequence", "label"]]
)
df_toxgin.shape

(4428, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_toxgin, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(4410, 2)

In [8]:
df_errors.shape

(9, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_toxgin)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2025, 1, 18, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'csv',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://github.com/cihebiyql/ToxGIN',
 'publication': 'https://academic.oup.com/bib/article/25/6/bbae583/7891575',
 'number_of_raw_sequences': 4428,
 'number_of_sequences_retained': 4410,
 'number_of_positive_sequences': 2205,
 'number_of_negative_sequences': 2205,
 'number_of_erroneous_sequences': 9,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)